# Datarock Broken Down Lead Deposit: geochemical proximity classification

**Objective:** use eight assay variables to classify samples as **A (proximal)** or **B (distal)** and predict the 767 intervals labelled `?`.

This notebook is the executable analysis and presentation deliverable. It covers QA/QC, drill-hole-aware validation, model comparison, per-hole evaluation, final scores, distribution checks, and the points that require domain-expert validation before trends are interpreted.

Key principles:

- validation holes are never present in the corresponding training fold;
- assay preprocessing is explicit and reproducible;
- model outputs are called **scores**, not calibrated probabilities;
- observed shifts are reported descriptively, without geological interpretation;
- final predictions are written to `output/predictions.csv`.

In [1]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

RANDOM_STATE = 42
ASSAY_FEATURES = ["As", "Au", "Pb", "Fe", "Mo", "Cu", "S", "Zn"]
META_COLUMNS = ["Unique_ID", "holeid", "from", "to"]
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "data_for_distribution.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "data_for_distribution.csv"
OUTPUT_DIR = PROJECT_ROOT / "output"
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 30)


def save_plot(filename):
    path = PLOT_DIR / filename
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved: {path}")

## 1. Load data and perform QA/QC

The QA/QC checks cover sample identifiers, interval geometry, missing values, non-numeric detection-limit strings, and the `-999` sentinel described by the challenge.

In [2]:
raw = pd.read_csv(DATA_PATH)

print(f"Rows: {len(raw):,}")
print(f"Unique sample IDs: {raw['Unique_ID'].nunique():,}")
print(f"Drill holes: {raw['holeid'].nunique():,}")
print("\nClass counts:")
display(raw["Class"].value_counts(dropna=False).rename("rows").to_frame())

assert raw["Unique_ID"].is_unique, "Unique_ID contains duplicates."
assert (pd.to_numeric(raw["to"]) > pd.to_numeric(raw["from"])).all(), (
    "Found non-positive intervals."
)

Rows: 4,771
Unique sample IDs: 4,771
Drill holes: 140

Class counts:


,rows
Class,
A,2861
B,1143
?,767


In [3]:
def raw_quality_counts(df):
    rows = []
    for column in ASSAY_FEATURES:
        values = df[column]
        text = values.astype("string").str.strip()
        numeric = pd.to_numeric(values, errors="coerce")
        rows.append({
            "feature": column,
            "blank_or_na": int(values.isna().sum()),
            "below_detection_strings": int(text.str.startswith("<", na=False).sum()),
            "sentinel_minus_999": int((numeric == -999).sum()),
        })
    return pd.DataFrame(rows).set_index("feature")


quality_counts = raw_quality_counts(raw)
display(quality_counts)

,blank_or_na,below_detection_strings,sentinel_minus_999
feature,,,
As,1503,0,0
Au,6,464,0
Pb,15,0,0
Fe,62,0,0
Mo,30,0,28
Cu,25,0,0
S,10,0,0
Zn,9,0,0


### Assay-cleaning assumption

Values such as `<0.005` indicate a censored measurement rather than an ordinary missing value. For this exercise:

> `<x` is replaced with `x / 2`.

The `-999` sentinel is treated as missing. The detection-limit convention is a modelling assumption that must be validated with an assay or geochemistry domain expert before operational use.

In [4]:
LESS_THAN_PATTERN = re.compile(
    r"^<\s*([0-9]*\.?[0-9]+(?:[eE][-+]?\d+)?)$"
)


def parse_assay_value(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, str):
        value = value.strip()
        match = LESS_THAN_PATTERN.match(value)
        if match:
            return float(match.group(1)) / 2.0

    numeric = pd.to_numeric(value, errors="coerce")
    if pd.isna(numeric) or numeric == -999:
        return np.nan
    return float(numeric)


clean = raw.copy()
for column in ASSAY_FEATURES:
    clean[column] = clean[column].map(parse_assay_value)

labelled = clean[clean["Class"].isin(["A", "B"])].copy()
unlabelled = clean[clean["Class"].eq("?")].copy()

print(f"Labelled samples: {len(labelled):,}")
print(f"Unlabelled samples: {len(unlabelled):,}")
print(f"Labelled holes: {labelled['holeid'].nunique()}")
print(f"Unlabelled holes: {unlabelled['holeid'].nunique()}")
print(
    "Train/prediction hole overlap: "
    f"{len(set(labelled['holeid']) & set(unlabelled['holeid']))}"
)

Labelled samples: 4,004
Unlabelled samples: 767
Labelled holes: 123
Unlabelled holes: 17
Train/prediction hole overlap: 0


In [5]:
def missingness_percent(df):
    return df[ASSAY_FEATURES].isna().mean().mul(100)


missingness = pd.DataFrame({
    "labelled_missing_%": missingness_percent(labelled),
    "unlabelled_missing_%": missingness_percent(unlabelled),
})
missingness["difference_pp"] = (
    missingness["unlabelled_missing_%"] - missingness["labelled_missing_%"]
)
missingness.to_csv(OUTPUT_DIR / "missingness_comparison.csv")
display(missingness.round(2))

ax = missingness[["labelled_missing_%", "unlabelled_missing_%"]].plot(
    kind="bar", figsize=(9, 4), color=["#295f98", "#e07a2d"]
)
ax.set_ylabel("Missing values (%)")
ax.set_title("Missingness in labelled and unlabelled datasets")
ax.legend(["Labelled", "Unlabelled"])
plt.xticks(rotation=0)
plt.tight_layout()
save_plot("missingness_comparison.png")

,labelled_missing_%,unlabelled_missing_%,difference_pp
As,36.19,7.04,-29.15
Au,0.12,0.13,0.01
Pb,0.32,0.26,-0.06
Fe,1.17,1.96,0.78
Mo,0.55,4.69,4.14
Cu,0.55,0.39,-0.16
S,0.25,0.00,-0.25
Zn,0.20,0.13,-0.07


Saved: /Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/output/plots/missingness_comparison.png


/var/folders/z_/qjbflxrj5vb73ny3sng4gwlw0000gn/T/ipykernel_576/3987348208.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The table and plot above are descriptive checks only. Differences between the labelled and unlabelled datasets do not establish a cause or geological meaning. Dataset comparability—particularly the large change in `As` missingness and the smaller change in `Mo` missingness—must be validated with domain experts before conclusions are drawn from downstream trends.

### Compact exploratory data analysis

The summary below combines the raw assay-quality counts with missingness by dataset and skewness calculated from the cleaned, non-missing labelled values before imputation. The multi-panel figure compares labelled and unlabelled distributions on a `log1p` scale so that long right tails remain visible without allowing the largest concentrations to dominate the display.

In [6]:
eda_summary = quality_counts.copy()
eda_summary["labelled_missing_%"] = missingness["labelled_missing_%"]
eda_summary["prediction_missing_%"] = missingness["unlabelled_missing_%"]
eda_summary["labelled_median"] = labelled[ASSAY_FEATURES].median()
eda_summary["labelled_skewness"] = labelled[ASSAY_FEATURES].skew()
eda_summary.to_csv(OUTPUT_DIR / "assay_eda_summary.csv")
display(eda_summary.round({
    "labelled_missing_%": 2,
    "prediction_missing_%": 2,
    "labelled_median": 3,
    "labelled_skewness": 2,
}))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feature in zip(axes.flat, ASSAY_FEATURES):
    labelled_log = np.log1p(labelled[feature].dropna())
    prediction_log = np.log1p(unlabelled[feature].dropna())
    combined_log = pd.concat([labelled_log, prediction_log], ignore_index=True)
    bins = np.linspace(combined_log.min(), combined_log.max(), 31)
    ax.hist(
        labelled_log, bins=bins, density=True, alpha=0.55,
        color="#295f98", label="Labelled",
    )
    ax.hist(
        prediction_log, bins=bins, density=True, alpha=0.45,
        color="#e07a2d", label="Unlabelled",
    )
    ax.set_title(feature)
    ax.set_xlabel("log1p(assay value)")
    ax.set_ylabel("Density")

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.95),
    ncol=2, frameon=False,
)
fig.suptitle(
    "Cleaned assay distributions: labelled vs unlabelled intervals",
    y=0.995, fontsize=15,
)
fig.text(
    0.5, 0.01,
    "Density histograms on a log1p scale; missing values excluded",
    ha="center", fontsize=10,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.88])
save_plot("assay_distributions.png")

,blank_or_na,below_detection_strings,sentinel_minus_999,labelled_missing_%,prediction_missing_%,labelled_median,labelled_skewness
feature,,,,,,,
As,1503,0,0,36.19,7.04,8.800,9.68
Au,6,464,0,0.12,0.13,0.028,7.51
Pb,15,0,0,0.32,0.26,463.800,9.68
Fe,62,0,0,1.17,1.96,48760.000,3.70
Mo,30,0,28,0.55,4.69,5.600,25.82
Cu,25,0,0,0.55,0.39,5.000,49.21
S,10,0,0,0.25,0.00,3740.000,3.93
Zn,9,0,0,0.20,0.13,37.600,12.78


Saved: /Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/output/plots/assay_distributions.png


/var/folders/z_/qjbflxrj5vb73ny3sng4gwlw0000gn/T/ipykernel_576/3987348208.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


All eight labelled assay variables have strong positive skew on the original scale, supporting median rather than mean imputation as a robust baseline. This does not establish that median imputation is geochemically correct. The missingness mechanisms, assay units, analytical methods, detection limits, and differences between labelled and unlabelled distributions require validation by geochemistry, assay QA/QC, and operational domain experts. No geological interpretation is assigned to these descriptive distributions.

## 2. Prepare assay features and drill-hole-aware validation

The model uses only the eight assay variables. Sample identifiers, hole identifiers, and interval depths are retained as metadata but excluded from the predictors.

Adjacent intervals from the same hole are related. A row-random split could therefore place neighbouring intervals from one hole in both training and validation. `StratifiedGroupKFold` keeps every validation hole completely unseen during training.

In [7]:
X = labelled[ASSAY_FEATURES]
y = labelled["Class"].map({"B": 0, "A": 1}).astype(int)
groups = labelled["holeid"]
X_future = unlabelled[ASSAY_FEATURES]

class_proportions = labelled["Class"].value_counts(normalize=True).rename(
    "proportion"
)
print("Labelled class proportions:")
display(class_proportions.round(3).to_frame())

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_assignments = np.full(len(labelled), -1, dtype=int)
for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    train_holes = set(groups.iloc[train_idx])
    valid_holes = set(groups.iloc[valid_idx])
    assert train_holes.isdisjoint(valid_holes)
    fold_assignments[valid_idx] = fold
    print(
        f"Fold {fold}: "
        f"{len(train_idx):,} train rows / {len(valid_idx):,} validation rows, "
        f"{len(train_holes)} train holes / {len(valid_holes)} validation holes"
    )

assert (fold_assignments > 0).all()

Labelled class proportions:


,proportion
Class,
A,0.715
B,0.285


Fold 1: 3,250 train rows / 754 validation rows, 94 train holes / 29 validation holes
Fold 2: 3,279 train rows / 725 validation rows, 98 train holes / 25 validation holes
Fold 3: 3,027 train rows / 977 validation rows, 99 train holes / 24 validation holes
Fold 4: 3,056 train rows / 948 validation rows, 101 train holes / 22 validation holes
Fold 5: 3,404 train rows / 600 validation rows, 100 train holes / 23 validation holes


## 3. Compare an interpretable baseline with a nonlinear model

The logistic baseline uses log-transformed, standardised assay values. Extra Trees captures nonlinearities and interactions. Both pipelines impute medians, add missingness indicators, and use balanced class weights.

Model selection uses grouped out-of-fold **balanced accuracy**. ROC-AUC and positive-class precision, recall, and F1 are also reported.

In [8]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Extra Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("clf", ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
}


def grouped_oof_evaluation(model, X, y, groups, cv):
    oof_pred = np.zeros(len(y), dtype=int)
    oof_score = np.zeros(len(y), dtype=float)
    fold_metrics = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y, groups), start=1
    ):
        fitted = clone(model)
        fitted.fit(X.iloc[train_idx], y.iloc[train_idx])

        prediction = fitted.predict(X.iloc[valid_idx])
        score = fitted.predict_proba(X.iloc[valid_idx])[:, 1]

        oof_pred[valid_idx] = prediction
        oof_score[valid_idx] = score

        fold_metrics.append({
            "fold": fold,
            "validation_rows": len(valid_idx),
            "validation_holes": groups.iloc[valid_idx].nunique(),
            "accuracy": accuracy_score(y.iloc[valid_idx], prediction),
            "balanced_accuracy": balanced_accuracy_score(
                y.iloc[valid_idx], prediction
            ),
            "roc_auc": roc_auc_score(y.iloc[valid_idx], score),
            "precision_A": precision_score(
                y.iloc[valid_idx], prediction, zero_division=0
            ),
            "recall_A": recall_score(
                y.iloc[valid_idx], prediction, zero_division=0
            ),
            "f1_A": f1_score(y.iloc[valid_idx], prediction, zero_division=0),
        })

    summary = {
        "accuracy": accuracy_score(y, oof_pred),
        "balanced_accuracy": balanced_accuracy_score(y, oof_pred),
        "roc_auc": roc_auc_score(y, oof_score),
        "precision_A": precision_score(y, oof_pred, zero_division=0),
        "recall_A": recall_score(y, oof_pred, zero_division=0),
        "f1_A": f1_score(y, oof_pred, zero_division=0),
    }

    return summary, pd.DataFrame(fold_metrics), oof_pred, oof_score


summaries = []
evaluation_objects = {}

for name, model in models.items():
    summary, fold_metrics, oof_pred, oof_score = grouped_oof_evaluation(
        model, X, y, groups, cv
    )
    summaries.append({"model": name, **summary})
    evaluation_objects[name] = {
        "fold_metrics": fold_metrics,
        "oof_pred": oof_pred,
        "oof_score": oof_score,
    }

results = pd.DataFrame(summaries).sort_values(
    "balanced_accuracy", ascending=False
).reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
display(results.round(3))

/Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matm

,model,accuracy,balanced_accuracy,roc_auc,precision_A,recall_A,f1_A
0,Extra Trees,0.801,0.793,0.867,0.900,0.811,0.853
1,Logistic Regression,0.772,0.770,0.835,0.892,0.775,0.829


In [9]:
metric_columns = ["balanced_accuracy", "roc_auc", "f1_A"]
plot_values = results.set_index("model")[metric_columns]
ax = plot_values.plot(
    kind="bar", figsize=(9, 4.5), color=["#295f98", "#49a078", "#e07a2d"]
)
ax.set_ylim(0, 1)
ax.set_ylabel("Grouped out-of-fold score")
ax.set_title("Model comparison on unseen drill holes")
ax.legend(["Balanced accuracy", "ROC-AUC", "F1 for A"])
plt.xticks(rotation=0)
plt.tight_layout()
save_plot("model_comparison.png")

Saved: /Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/output/plots/model_comparison.png


/var/folders/z_/qjbflxrj5vb73ny3sng4gwlw0000gn/T/ipykernel_576/3987348208.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
best_name = results.iloc[0]["model"]
best_oof = evaluation_objects[best_name]
best_oof["fold_metrics"].to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)

print(f"Selected model: {best_name}")
print("\nFold-to-fold validation results:")
display(best_oof["fold_metrics"].round(3))

cm = confusion_matrix(y, best_oof["oof_pred"], labels=[1, 0])
cm_df = pd.DataFrame(
    cm,
    index=["Actual A", "Actual B"],
    columns=["Predicted A", "Predicted B"],
)
display(cm_df)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
image = ax.imshow(cm, cmap="Blues")
for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        ax.text(column, row, f"{cm[row, column]:,}", ha="center", va="center")
ax.set_xticks([0, 1], ["Predicted A", "Predicted B"])
ax.set_yticks([0, 1], ["Actual A", "Actual B"])
ax.set_title(f"Grouped out-of-fold confusion matrix: {best_name}")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
save_plot("confusion_matrix.png")

Selected model: Extra Trees

Fold-to-fold validation results:


,fold,validation_rows,validation_holes,accuracy,balanced_accuracy,roc_auc,precision_A,recall_A,f1_A
0,1,754,29,0.763,0.731,0.808,0.902,0.785,0.839
1,2,725,25,0.808,0.787,0.862,0.938,0.820,0.875
2,3,977,24,0.791,0.818,0.908,0.936,0.739,0.826
3,4,948,22,0.813,0.809,0.890,0.807,0.862,0.834
4,5,600,23,0.835,0.757,0.850,0.924,0.874,0.898


,Predicted A,Predicted B
Actual A,2321,540
Actual B,258,885


Saved: /Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/output/plots/confusion_matrix.png


/var/folders/z_/qjbflxrj5vb73ny3sng4gwlw0000gn/T/ipykernel_576/3987348208.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Per-hole evaluation

Overall row-level metrics weight holes in proportion to their interval counts. The following table evaluates each validation hole separately so that performance variation across holes is visible.

Per-hole balanced accuracy and ROC-AUC are only defined here for holes containing both classes. Per-hole accuracy is available for all holes, including single-class holes. The mean per-hole accuracy gives each hole equal weight.

In [11]:
oof_rows = pd.DataFrame({
    "holeid": labelled["holeid"].to_numpy(),
    "fold": fold_assignments,
    "actual_A": y.to_numpy(),
    "predicted_A": best_oof["oof_pred"],
    "model_score_A": best_oof["oof_score"],
})

per_hole_rows = []
for holeid, hole_rows in oof_rows.groupby("holeid", sort=True):
    has_both_classes = hole_rows["actual_A"].nunique() == 2
    per_hole_rows.append({
        "holeid": holeid,
        "fold": int(hole_rows["fold"].iloc[0]),
        "intervals": len(hole_rows),
        "actual_A_rate": hole_rows["actual_A"].mean(),
        "predicted_A_rate": hole_rows["predicted_A"].mean(),
        "accuracy": accuracy_score(
            hole_rows["actual_A"], hole_rows["predicted_A"]
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                hole_rows["actual_A"], hole_rows["predicted_A"]
            ) if has_both_classes else np.nan
        ),
        "roc_auc": (
            roc_auc_score(
                hole_rows["actual_A"], hole_rows["model_score_A"]
            ) if has_both_classes else np.nan
        ),
        "mean_model_score_A": hole_rows["model_score_A"].mean(),
    })

per_hole = pd.DataFrame(per_hole_rows)
per_hole.to_csv(OUTPUT_DIR / "per_hole_validation.csv", index=False)

per_hole_summary = pd.DataFrame([{
    "holes": len(per_hole),
    "mean_hole_accuracy": per_hole["accuracy"].mean(),
    "median_hole_accuracy": per_hole["accuracy"].median(),
    "p10_hole_accuracy": per_hole["accuracy"].quantile(0.10),
    "p90_hole_accuracy": per_hole["accuracy"].quantile(0.90),
    "holes_with_both_classes": per_hole["balanced_accuracy"].notna().sum(),
    "mean_balanced_accuracy_two_class_holes": (
        per_hole["balanced_accuracy"].mean()
    ),
    "mean_roc_auc_two_class_holes": per_hole["roc_auc"].mean(),
}])
per_hole_summary.to_csv(OUTPUT_DIR / "per_hole_summary.csv", index=False)

display(per_hole_summary.round(3))
print("Lowest per-hole validation accuracies:")
display(per_hole.nsmallest(10, "accuracy").round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(per_hole["accuracy"], bins=np.linspace(0, 1, 11), color="#295f98", edgecolor="white")
ax.axvline(
    per_hole["accuracy"].mean(),
    color="#e07a2d",
    linestyle="--",
    label=f"Mean per-hole accuracy = {per_hole['accuracy'].mean():.3f}",
)
ax.set_xlabel("Validation accuracy within each hole")
ax.set_ylabel("Number of holes")
ax.set_title("Distribution of validation accuracy across drill holes")
ax.legend()
plt.tight_layout()
save_plot("per_hole_accuracy.png")

,holes,mean_hole_accuracy,median_hole_accuracy,p10_hole_accuracy,p90_hole_accuracy,holes_with_both_classes,mean_balanced_accuracy_two_class_holes,mean_roc_auc_two_class_holes
0,123,0.872,0.958,0.659,1.0,32,0.715,0.809


Lowest per-hole validation accuracies:


,holeid,fold,intervals,actual_A_rate,predicted_A_rate,accuracy,balanced_accuracy,roc_auc,mean_model_score_A
8,SOLVE017,3,10,1.000,0.000,0.000,NaN,NaN,0.315
90,SOLVE157W1,2,30,1.000,0.233,0.233,NaN,NaN,0.455
9,SOLVE021,1,33,1.000,0.485,0.485,NaN,NaN,0.481
46,SOLVE080,1,103,0.602,0.534,0.485,0.478,0.529,0.553
30,SOLVE055,4,2,1.000,0.500,0.500,NaN,NaN,0.537
16,SOLVE036,1,28,1.000,0.536,0.536,NaN,NaN,0.507
89,SOLVE157,5,39,1.000,0.590,0.590,NaN,NaN,0.513
97,SOLVE171,1,32,1.000,0.594,0.594,NaN,NaN,0.553
24,SOLVE044,5,77,0.519,0.584,0.597,0.594,0.636,0.551
91,SOLVE158,3,65,1.000,0.600,0.600,NaN,NaN,0.546


Saved: /Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/output/plots/per_hole_accuracy.png


/var/folders/z_/qjbflxrj5vb73ny3sng4gwlw0000gn/T/ipykernel_576/3987348208.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The per-hole spread is a validation result, not an explanation of why particular holes perform differently. Any relationship between difficult holes and geology, sampling, assay campaigns, or spatial context requires domain-expert review.

## 5. Refit on all labelled samples and score unseen holes

The selected model is refit on all 4,004 labelled intervals. `predict_proba` from Extra Trees is exposed as `model_score_A`, explicitly **not** as a calibrated probability. `score_strength` is the distance toward the selected class, and `near_decision_boundary` flags scores between 0.40 and 0.60.

In [12]:
final_model = clone(models[best_name])
final_model.fit(X, y)

model_score_A = final_model.predict_proba(X_future)[:, 1]
predicted_class = np.where(model_score_A >= 0.5, "A", "B")
score_strength = np.maximum(model_score_A, 1 - model_score_A)

predictions = raw.loc[
    unlabelled.index, META_COLUMNS + ASSAY_FEATURES
].copy()
predictions["predicted_class"] = predicted_class
predictions["model_score_A"] = model_score_A
predictions["score_strength"] = score_strength
predictions["near_decision_boundary"] = score_strength < 0.60

prediction_path = OUTPUT_DIR / "predictions.csv"
predictions.to_csv(prediction_path, index=False)

assert len(predictions) == 767
assert predictions["Unique_ID"].is_unique
assert predictions["predicted_class"].isin(["A", "B"]).all()
assert predictions["model_score_A"].between(0, 1).all()

print("Predicted class counts:")
display(predictions["predicted_class"].value_counts().rename("rows").to_frame())
print(
    "Near-decision-boundary scores (0.40 < score_A < 0.60): "
    f"{predictions['near_decision_boundary'].sum():,}"
)
print(f"Saved: {prediction_path}")

Predicted class counts:


,rows
predicted_class,
B,500
A,267


Near-decision-boundary scores (0.40 < score_A < 0.60): 133
Saved: /Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/output/predictions.csv


## 6. Compare labelled and prediction class distributions

This comparison reports observed class proportions only. The predicted share of A in new holes is a model output, not a known class distribution, and it must not be treated as a geological conclusion without validation.

In [13]:
labelled_A_rate = labelled["Class"].eq("A").mean()
predicted_A_rate = predictions["predicted_class"].eq("A").mean()

class_distribution = pd.DataFrame({
    "dataset": ["Labelled intervals", "Prediction intervals"],
    "A_rate": [labelled_A_rate, predicted_A_rate],
    "B_rate": [1 - labelled_A_rate, 1 - predicted_A_rate],
})
class_distribution["A_rate_difference_vs_labelled_pp"] = [
    0.0,
    100 * (predicted_A_rate - labelled_A_rate),
]
class_distribution.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)
display(class_distribution.round(3))

ax = class_distribution.set_index("dataset")[["A_rate", "B_rate"]].plot(
    kind="bar", stacked=True, figsize=(8, 4.5), color=["#c94c4c", "#d9d9d9"]
)
ax.set_ylim(0, 1)
ax.set_ylabel("Share of intervals")
ax.set_title("Observed labelled classes and model-predicted classes")
ax.legend(["Class A", "Class B"])
plt.xticks(rotation=0)
plt.tight_layout()
save_plot("class_distribution.png")

,dataset,A_rate,B_rate,A_rate_difference_vs_labelled_pp
0,Labelled intervals,0.715,0.285,0.000
1,Prediction intervals,0.348,0.652,-36.643


Saved: /Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/output/plots/class_distribution.png


/var/folders/z_/qjbflxrj5vb73ny3sng4gwlw0000gn/T/ipykernel_576/3987348208.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The observed difference is substantial: Class A represents about 71.5% of labelled intervals, while the model assigns Class A to about 34.8% of prediction intervals. This result is not interpreted here.

Before drawing conclusions from this trend, domain experts should validate:

1. whether the labelled and new holes are comparable in geological setting, drilling purpose, assay campaign, laboratory, and sampling process;
2. whether a different proximal/distal prevalence is expected in the new drilling area;
3. whether the missingness differences—especially for `As` and `Mo`—reflect benign acquisition changes or a material dataset shift;
4. whether the strongest predictive associations are plausible or are dataset-specific proxies;
5. whether a 0.50 score threshold reflects the operational costs of false proximal and false distal calls.

Until those checks are complete, the class-distribution difference is a result requiring validation rather than evidence for a geological explanation.

## 7. Inspect predictive associations

Feature importance is reported as predictive association only. It does not establish geological causation.

In [14]:
imputer = final_model.named_steps["imputer"]
classifier = final_model.named_steps["clf"]
feature_names = imputer.get_feature_names_out(ASSAY_FEATURES)
importance = pd.Series(
    classifier.feature_importances_, index=feature_names, name="importance"
).sort_values(ascending=False)
importance.to_csv(OUTPUT_DIR / "feature_importance.csv")
display(importance.head(12).to_frame())

ax = importance.head(12).sort_values().plot(
    kind="barh", figsize=(8, 5), color="#295f98"
)
ax.set_title("Extra Trees predictive feature importance")
ax.set_xlabel("Impurity-based importance")
plt.tight_layout()
save_plot("feature_importance.png")

,importance
Pb,0.298674
Mo,0.187521
As,0.134713
Au,0.124344
Fe,0.079979
S,0.079069
Zn,0.039477
Cu,0.024643
missingindicator_As,0.018080
missingindicator_Fe,0.004503


Saved: /Users/abiha07/Documents/Codex/2026-08-11/run/work/datarock-code-challenge/output/plots/feature_importance.png


/var/folders/z_/qjbflxrj5vb73ny3sng4gwlw0000gn/T/ipykernel_576/3987348208.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The ranking above should be reviewed with geology and geochemistry experts before any meaning is assigned to the observed associations.

## 8. Downhole continuity check

Intervals are scored independently. The summary below counts predicted A/B transitions within each new hole; it does not smooth or reinterpret the predictions.

In [15]:
def count_transitions(series):
    values = series.to_numpy()
    if len(values) < 2:
        return 0
    return int(np.sum(values[1:] != values[:-1]))


continuity = (
    predictions
    .sort_values(["holeid", "from"])
    .groupby("holeid")
    .agg(
        intervals=("predicted_class", "size"),
        class_transitions=("predicted_class", count_transitions),
        mean_model_score_A=("model_score_A", "mean"),
    )
    .sort_values(["class_transitions", "intervals"], ascending=False)
)
continuity.to_csv(OUTPUT_DIR / "downhole_continuity.csv")
display(continuity.head(10).round(3))

,intervals,class_transitions,mean_model_score_A
holeid,,,
SOLVE236,164,24,0.494
SOLVE291,96,21,0.534
SOLVE279A,163,7,0.487
SOLVE237,154,6,0.293
SOLVE225W1,26,2,0.756
SOLVE225W2,18,2,0.697
SOLVE233,10,2,0.388
SOLVE232,14,1,0.311
SOLVE278,24,0,0.761


No smoothing is applied because an expected continuity scale has not been established. Domain experts must validate whether the observed switching rates are plausible before a spatial, sequence, or smoothing assumption is introduced.

## 9. Results summary and validation boundary

The reproducible outputs establish the following model results:

- Extra Trees is selected over logistic regression using grouped out-of-fold balanced accuracy.
- The final model is trained on all labelled intervals and produces scores for all 767 unlabelled intervals.
- Per-hole metrics show how validation performance varies when each drill hole is inspected separately.
- The labelled/prediction missingness and class-distribution differences are recorded explicitly.

These results support a predictive prototype. They do **not** by themselves support conclusions about geological processes, assay-program causes, expected prevalence in future drilling, or operational decision thresholds. Those parts require domain-expert validation before any trend is interpreted or operationalised.